In [2]:
import torch as torch

In [68]:
import torch
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)

In [5]:
inputs.shape

torch.Size([6, 3])

In [7]:
x_2 = inputs[1]
in_dim = inputs.shape[1]
w_size = 2

In [31]:
W_query = torch.nn.Parameter(torch.rand(in_dim, w_size))
W_key = torch.nn.Parameter(torch.rand(in_dim, w_size))
W_value = torch.nn.Parameter(torch.rand(in_dim, w_size))

K = inputs @ W_key
Q = inputs @ W_query
V = inputs @ W_value

print(Q)

tensor([[0.4173, 0.4396],
        [0.4395, 0.8128],
        [0.4299, 0.8047],
        [0.2369, 0.4641],
        [0.1370, 0.4310],
        [0.3458, 0.5718]], grad_fn=<MmBackward0>)


In [11]:
d_k = K.shape[1]
att_score = torch.softmax(Q @ K.T/d_k**0.5, dim=1)

output = att_score @ V
output

tensor([[-0.3685,  0.3186],
        [-0.3598,  0.3162],
        [-0.3601,  0.3169],
        [-0.3800,  0.3383],
        [-0.3714,  0.3402],
        [-0.3774,  0.3305]], grad_fn=<MmBackward0>)

# Implementing an Attention Class

In [69]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, in_dim, w_size):
        super().__init__()
        self.W_query = torch.nn.Parameter(torch.randn(in_dim, w_size))
        self.W_key = torch.nn.Parameter(torch.randn(in_dim, w_size))
        self.W_value = torch.nn.Parameter(torch.randn(in_dim, w_size))

    def forward(self, inputs):
        K = inputs @ self.W_key
        Q = inputs @ self.W_query
        V = inputs @ self.W_value

        d_k = K.shape[-1]
        att_score = torch.softmax(Q @ K.T/d_k**0.5, dim=1)

        output = att_score @ V
        return output

In [70]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(in_dim, w_size)
print(sa_v1(inputs))

tensor([[0.2845, 0.4071],
        [0.2854, 0.4081],
        [0.2854, 0.4075],
        [0.2864, 0.3974],
        [0.2863, 0.3910],
        [0.2860, 0.4039]], grad_fn=<MmBackward0>)


In [71]:
import torch.nn as nn

class SelfAttention_v2(nn.Module):
    def __init__(self, in_dim, w_size):
        super().__init__()
        self.W_query = torch.nn.Linear(in_dim, w_size, bias=False)
        self.W_key = torch.nn.Linear(in_dim, w_size, bias=False)
        self.W_value = torch.nn.Linear(in_dim, w_size, bias=False)

    def forward(self, inputs):
        K = self.W_key(inputs)
        Q = self.W_query(inputs)
        V = self.W_value(inputs)

        d_k = K.shape[-1]
        att_score_prev = Q @ K.T

        mask = torch.triu(torch.ones(inputs.shape[-2], inputs.shape[-2]), diagonal=1)
        masked = att_score_prev.masked_fill(mask.bool(), -torch.inf)

        att_score = torch.softmax(masked/d_k**0.5, dim=1)

        output = att_score @ V
        return output

In [72]:
torch.manual_seed(789)
sa_v1 = SelfAttention_v2(in_dim, w_size)
print(sa_v1(inputs))

tensor([[-0.0872,  0.0286],
        [-0.0991,  0.0501],
        [-0.0999,  0.0633],
        [-0.0983,  0.0489],
        [-0.0514,  0.1098],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


Original version

In [ ]:
import torch.nn as nn

class SelfAttention_v3(nn.Module):
    def __init__(self, in_dim, w_size, dropout):
        super().__init__()
        self.W_query = torch.nn.Linear(in_dim, w_size, bias=False)
        self.W_key = torch.nn.Linear(in_dim, w_size, bias=False)
        self.W_value = torch.nn.Linear(in_dim, w_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs):
        K = self.W_key(inputs)
        Q = self.W_query(inputs)
        V = self.W_value(inputs)

        d_k = K.shape[-1]
        att_score_prev = Q @ torch.transpose(K, -2, -1)

        mask = torch.triu(torch.ones(inputs.shape[-2], inputs.shape[-2]), diagonal=1)
        masked = att_score_prev.masked_fill(mask.bool(), -torch.inf)

        att_score = torch.softmax(masked/d_k**0.5, dim=1)
        self.dropout(att_score)

        output = att_score @ V
        return output

Corrected One

In [78]:
import torch.nn as nn

class SelfAttention_v3(nn.Module):
    def __init__(self, in_dim, w_size, dropout, context_lenght):
        super().__init__()
        self.W_query = torch.nn.Linear(in_dim, w_size, bias=False)
        self.W_key = torch.nn.Linear(in_dim, w_size, bias=False)
        self.W_value = torch.nn.Linear(in_dim, w_size, bias=False)
        self.dropout = nn.Dropout(dropout)

        # We use a buffer to move the mask automatically to torch.device
        self.register_buffer('mask', torch.triu(torch.ones(context_lenght, context_lenght), diagonal=1))

    def forward(self, inputs):
        K = self.W_key(inputs)
        Q = self.W_query(inputs)
        V = self.W_value(inputs)

        d_k = K.shape[-1]
        att_score_prev = Q @ torch.transpose(K, -2, -1)

        # Use masked_fill_ for inplace opperation
        masked = att_score_prev.masked_fill_(self.mask.bool(), -torch.inf)

        # Dim not 1. Is -1. (batch size, rows, cols). We want to apply to rows, ergo, 'remove' the cols.
        att_score = torch.softmax(masked/d_k**0.5, dim=-1)

        # Dropout not in place, store the layer
        att_score = self.dropout(att_score)

        output = att_score @ V
        return output

In [79]:
inputs = torch.tensor([
    [[0.43, 0.15, 0.89],
     [0.55, 0.87, 0.66],
     [0.57, 0.85, 0.64]],

    [[0.22, 0.58, 0.33],
     [0.77, 0.25, 0.10],
     [0.05, 0.80, 0.55]],

    [[0.31, 0.42, 0.73],
     [0.64, 0.29, 0.91],
     [0.18, 0.56, 0.47]],

    [[0.82, 0.13, 0.35],
     [0.46, 0.72, 0.21],
     [0.93, 0.38, 0.60]],

    [[0.25, 0.67, 0.49],
     [0.71, 0.32, 0.84],
     [0.14, 0.95, 0.28]],

    [[0.52, 0.76, 0.19],
     [0.36, 0.88, 0.41],
     [0.69, 0.23, 0.57]]
])

print(inputs.shape)

torch.Size([6, 3, 3])


In [83]:
torch.manual_seed(789)
in_dim, w_size, dropout, context_length = 3, 2, 0.1, 3
sa_v1 = SelfAttention_v3(in_dim, w_size, dropout, context_length)
print(sa_v1(inputs))
print(sa_v1(inputs).shape)

tensor([[[-0.0969,  0.0318],
         [-0.1101,  0.0557],
         [-0.1111,  0.0704]],

        [[-0.1014, -0.0029],
         [ 0.0296,  0.2017],
         [ 0.0215,  0.1389]],

        [[-0.1382, -0.0267],
         [-0.0988,  0.0595],
         [-0.1119,  0.0232]],

        [[ 0.1334,  0.3812],
         [ 0.0579,  0.2740],
         [ 0.0256,  0.1324]],

        [[-0.1420, -0.0305],
         [-0.0191,  0.1057],
         [-0.1086,  0.0423]],

        [[-0.0150,  0.1849],
         [-0.0712,  0.1075],
         [-0.0319,  0.1600]]], grad_fn=<UnsafeViewBackward0>)
torch.Size([6, 3, 2])


# Multihead Attention

In [89]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads):
        super().__init__()
        self.heads = nn.ModuleList([SelfAttention_v3(d_in, d_out, dropout, context_length) for _ in range(num_heads)])

    def forward(self, input):
        return torch.cat([head(input) for head in self.heads], dim=-1)

In [92]:
torch.manual_seed(123)

inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)
batch = torch.stack((inputs, inputs), dim=0)

context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
d_in, d_out, context_length, 0.0, num_heads=2
)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


In [ ]:
class MultiHeadAttention(nn.Module):
    #d_in = embeding size, d_out = head_dim * num_heads
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, ):
        super().__init__()
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)

        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", 
            torch.triu(torch.ones(context_length, context_length),
            diagonal=1)
            )
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1)
        
        attn_weights = self.dropout(attn_weights)
        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(
            b, num_tokens, self.d_out
            )
        context_vec = self.out_proj(context_vec)
        return context_vec